# Agreement of LLM Judges in Persona Contradiction Detection

**Author:** Matyáš Martinek
**Course:** Probability and Statistics
**Language:** Python

## Objective

This project investigates whether different large language models agree when
judging contradictions between a complete persona and a dialogue.

Each LLM judge receives the same persona–dialogue example and predicts one of
two labels:

- `CONTRADICTION`
- `NO_CONTRADICTION`

The primary goal is to measure inter-judge agreement and determine whether
different LLM judges exhibit systematically different decision behaviour.

The project further investigates whether agreement is related to properties
of the input, including persona length, dialogue length, lexical overlap, and
the number of augmented persona facts.

## 1. Data source and provenance

The base data come from the **PersonaChat** dataset introduced by
Zhang et al. (2018). PersonaChat contains multi-turn English dialogues in
which each participant is assigned a persona represented by several natural
language statements.

For this project, I use the PersonaChat distribution available on Hugging Face:

https://huggingface.co/datasets/awsaf49/persona-chat

The original dataset is described in:

> Zhang, S., Dinan, E., Urbanek, J., Szlam, A., Kiela, D., & Weston, J.
> (2018). *Personalizing Dialogue Agents: I have a dog, do you have pets too?*
> Proceedings of ACL 2018, pp. 2204–2213.
> DOI: 10.18653/v1/P18-1205.

Original paper:

https://aclanthology.org/P18-1205/

### Derived data

The PersonaChat data were processed as part of a bachelor-thesis project on
persona consistency.

Persona facts were automatically grounded in their corresponding dialogues.
Grounding estimates whether a persona fact is supported by evidence in the
dialogue.

Contradiction-oriented persona variants were subsequently generated using
deterministic rule-based augmentations. Multiple supported facts belonging to
the same persona may be modified simultaneously while the corresponding
dialogue remains unchanged.

The resulting profile-level dataset is stored in:

`personachat_augmented_v2.parquet`

Each row represents one speaker persona and its dialogue and contains:

- the complete original persona,
- the complete augmented persona,
- the unchanged dialogue,
- the number and types of applied augmentations,
- grounding information for the changed facts,
- augmentation provenance.

For rule-based profiles, an augmented persona is assigned the expected relation
`contradiction` when at least one changed persona fact is grounded in the
dialogue.

These automatically derived relations are treated as **expected labels**, not
as manually verified ground truth.

In [22]:
from pathlib import Path

import pandas as pd


RANDOM_SEED = 42

DATA_DIR = Path("data")

SOURCE_DATA_PATH = (
    DATA_DIR
    / "personachat_augmented_v2.parquet"
)

source_df = pd.read_parquet(
    SOURCE_DATA_PATH
)

print(f"Rows: {len(source_df):,}")
print(
    f"Dialogues: "
    f"{source_df['dialogue_id'].nunique():,}"
)

print(
    "\nAugmentation families:"
)

print(
    source_df[
        "augmentation_family"
    ].value_counts()
)

source_df.head()

Rows: 26,368
Dialogues: 8,939

Augmentation families:
augmentation_family
persona_swap    17878
rule             8490
Name: count, dtype: int64


,augmentation_id,dialogue_id,speaker,augmentation_family,augmentation_type,augmentation_method,dialogue_text,original_persona,augmented_persona,persona_fact_count,...,contradiction_strengths,generation_notes,rule_versions,changed_expected_relations,changed_grounding_labels,changed_grounding_scores,changed_best_utterances,expected_relation,donor_dialogue_id,seed
0,aug_000000,0,speaker_1,rule,preference_negation,rule_based,"SELF: hi , how are you doing ? i am getting re...","[i like canning and whittling., to stay in sha...","[i do not like canning and whittling., to stay...",4,...,[strong],[Preference polarity was negated using a deter...,[v1],[contradiction],[grounded],[0.9796050786972046],[i am ! for my hobby i like to do canning or s...,contradiction,NaN,42
1,aug_000001,0,speaker_2,rule,preference_negation,rule_based,"PARTNER: hi , how are you doing ? i am getting...","[i like to remodel homes., i like to go huntin...","[i do not like to remodel homes., i do not lik...",4,...,"[strong, strong]",[Preference polarity was negated using a deter...,"[v1, v1]","[contradiction, contradiction]","[grounded, grounded]","[0.8077989220619202, 0.8201733231544495]",[i also remodel homes when i am not out bow hu...,contradiction,NaN,42
2,aug_000002,1,speaker_1,rule,preference_negation,rule_based,"SELF: hi , how are you doing today ?\nPARTNER:...","[i wish i could live forever., i only date peo...","[i wish i could live forever., i only date peo...",4,...,[strong],[Preference polarity was negated using a deter...,[v1],[contradiction],[grounded],[0.9774878025054932],"[i really enjoy free diving , how about you , ...",contradiction,NaN,42
3,aug_000003,1,speaker_2,rule,family_composition_swap,rule_based,"PARTNER: hi , how are you doing today ?\nSELF:...","[my mom is my best friend., i have four sister...","[my mom is my best friend., i have 2 sisters.,...",4,...,[medium_strong],[Family composition count was replaced with a ...,[v1],[contradiction],[grounded],[0.9484180808067322],[i am spending time with my 4 sisters what are...,contradiction,NaN,42
4,aug_000004,2,speaker_1,rule,preference_negation,rule_based,"SELF: we all live in a yellow submarine , a ye...","[i love the beatles., i have trouble getting a...","[i hate the beatles., i have trouble getting a...",4,...,[strong],[Preference polarity was negated using a deter...,[v1],[contradiction],[grounded],[0.8553723692893982],"[lol . i am shy , anything to break the ice , ...",contradiction,NaN,42


In [23]:
eligible_df = source_df[
    source_df["augmentation_family"].eq("rule")
    & source_df["expected_relation"].eq("contradiction")
    & source_df["num_grounded_augmented_facts"].gt(0)
].copy()


print(
    f"Eligible rule-augmented profiles: "
    f"{len(eligible_df):,}"
)

print(
    f"Unique dialogues: "
    f"{eligible_df['dialogue_id'].nunique():,}"
)

Eligible rule-augmented profiles: 8,469
Unique dialogues: 6,399


In [24]:
profile_summary = pd.DataFrame(
    {
        "num_augmented_facts": (
            eligible_df[
                "num_augmented_facts"
            ].describe()
        ),
        "num_grounded_augmented_facts": (
            eligible_df[
                "num_grounded_augmented_facts"
            ].describe()
        ),
    }
)

profile_summary

,num_augmented_facts,num_grounded_augmented_facts
count,8469.000000,8469.000000
mean,1.502775,1.499351
std,0.740965,0.739694
min,1.000000,1.000000
25%,1.000000,1.000000
50%,1.000000,1.000000
75%,2.000000,2.000000
max,5.000000,5.000000


## 2. Experimental dataset construction

The experiment is based on complete persona–dialogue profiles rather than
individual persona facts.

For every selected profile, two matched evaluation examples are constructed:

1. **Original variant** – the complete original persona is paired with its
   dialogue and is expected to be `NO_CONTRADICTION`.
2. **Augmented variant** – the complete augmented persona is paired with the
   same dialogue and is expected to be `CONTRADICTION`.

The dialogue is therefore identical within each pair. The only difference is
that one or more persona statements have been modified in the augmented
variant.

Only rule-based profiles with expected relation `contradiction` are used. Such
profiles contain at least one changed persona fact that was grounded in the
dialogue.

To reduce dependence between observations, at most one speaker profile is
selected from each dialogue. Sampling is deterministic using a fixed random
seed.

Persona-swap examples are excluded from the main experiment because they are
currently contradiction candidates rather than verified contradiction-oriented
profiles.

In [25]:
N_PAIRS = 1000

In [26]:
sampling_pool = (
    eligible_df
    .sample(
        frac=1,
        random_state=RANDOM_SEED,
    )
    .drop_duplicates(
        subset="dialogue_id"
    )
    .reset_index(drop=True)
)

if len(sampling_pool) < N_PAIRS:
    raise ValueError(
        f"Only {len(sampling_pool):,} unique dialogues "
        f"are available, but {N_PAIRS:,} pairs were requested."
    )

selected_pairs_df = (
    sampling_pool
    .head(N_PAIRS)
    .copy()
)

print(
    f"Selected profiles: "
    f"{len(selected_pairs_df):,}"
)

print(
    f"Unique dialogues: "
    f"{selected_pairs_df['dialogue_id'].nunique():,}"
)

Selected profiles: 1,000
Unique dialogues: 1,000


In [27]:
def format_persona(
    persona,
) -> str:
    """
    Format a list of persona statements for LLM evaluation.
    """
    return "\n".join(
        f"- {fact}"
        for fact in persona
    )

In [29]:
original_df = selected_pairs_df.copy()

original_df["variant"] = "original"
original_df["persona_text"] = (
    original_df[
        "original_persona"
    ].map(format_persona)
)

original_df["expected_label"] = (
    "NO_CONTRADICTION"
)

original_df["expected_label_binary"] = 0


augmented_df = selected_pairs_df.copy()

augmented_df["variant"] = "augmented"
augmented_df["persona_text"] = (
    augmented_df[
        "augmented_persona"
    ].map(format_persona)
)

augmented_df["expected_label"] = (
    "CONTRADICTION"
)

augmented_df["expected_label_binary"] = 1

In [30]:
pair_ids = [
    f"pair_{i:04d}"
    for i in range(
        len(selected_pairs_df)
    )
]

original_df["pair_id"] = pair_ids
augmented_df["pair_id"] = pair_ids

In [31]:
experiment_df = pd.concat(
    [
        original_df,
        augmented_df,
    ],
    ignore_index=True,
)

experiment_df["example_id"] = (
    experiment_df["pair_id"]
    + "_"
    + experiment_df["variant"]
)

In [32]:
experiment_df = (
    experiment_df
    .sample(
        frac=1,
        random_state=RANDOM_SEED,
    )
    .reset_index(drop=True)
)

In [33]:
assert (
    len(experiment_df)
    == 2 * len(selected_pairs_df)
)

assert experiment_df[
    "example_id"
].is_unique

assert (
    experiment_df
    .groupby("pair_id")
    .size()
    .eq(2)
    .all()
)

assert (
    experiment_df
    .groupby("pair_id")[
        "dialogue_text"
    ]
    .nunique()
    .eq(1)
    .all()
)

assert (
    experiment_df
    .groupby("pair_id")[
        "dialogue_id"
    ]
    .nunique()
    .eq(1)
    .all()
)

assert (
    experiment_df[
        "pair_id"
    ].nunique()
    == experiment_df[
        "dialogue_id"
    ].nunique()
)

print(
    "Experimental dataset validation passed."
)

Experimental dataset validation passed.


### 2.1 Input characteristics

Several simple textual characteristics are computed before LLM evaluation.
These variables are later used to investigate whether judge decisions or
inter-judge disagreement depend on properties of the input.

The following characteristics are considered:

- complete persona length,
- number of persona facts,
- dialogue length,
- total textual input length,
- lexical overlap between the complete persona and the dialogue.

For augmented profiles, the source data additionally provide the number of
changed persona facts and the number of changed facts grounded in the dialogue.

Lengths are measured in word tokens using a simple deterministic tokenizer.
Lexical overlap is measured using Jaccard similarity between the sets of words
appearing in the persona and dialogue.

In [34]:
import re


WORD_PATTERN = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")


def tokenize_words(text: str) -> list[str]:
    """Return lowercase word tokens from English text."""
    return WORD_PATTERN.findall(str(text).lower())


def jaccard_overlap(text_a: str, text_b: str) -> float:
    """Compute Jaccard similarity between unique word sets."""
    words_a = set(tokenize_words(text_a))
    words_b = set(tokenize_words(text_b))

    union = words_a | words_b

    if not union:
        return 0.0

    return len(words_a & words_b) / len(union)

In [35]:
experiment_df[
    "persona_length"
] = (
    experiment_df[
        "persona_text"
    ]
    .map(tokenize_words)
    .map(len)
)

experiment_df[
    "dialogue_length"
] = (
    experiment_df[
        "dialogue_text"
    ]
    .map(tokenize_words)
    .map(len)
)

experiment_df[
    "input_length"
] = (
    experiment_df[
        "persona_length"
    ]
    + experiment_df[
        "dialogue_length"
    ]
)

experiment_df[
    "lexical_overlap"
] = experiment_df.apply(
    lambda row: jaccard_overlap(
        row["persona_text"],
        row["dialogue_text"],
    ),
    axis=1,
)

experiment_df[
    [
        "persona_fact_count",
        "persona_length",
        "dialogue_length",
        "input_length",
        "lexical_overlap",
        "num_augmented_facts",
        "num_grounded_augmented_facts",
    ]
].describe()

,persona_fact_count,persona_length,dialogue_length,input_length,lexical_overlap,num_augmented_facts,num_grounded_augmented_facts
count,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000
mean,4.540000,26.821500,165.115000,191.936500,0.141222,1.494000,1.491000
std,0.508458,6.706648,28.514247,29.638061,0.043630,0.687168,0.685678
min,3.000000,11.000000,74.000000,99.000000,0.025424,1.000000,1.000000
25%,4.000000,22.000000,146.000000,172.000000,0.111111,1.000000,1.000000
50%,5.000000,26.000000,164.000000,192.000000,0.135593,1.000000,1.000000
75%,5.000000,31.000000,184.000000,210.000000,0.165158,2.000000,2.000000
max,5.000000,67.000000,329.000000,352.000000,0.328947,4.000000,4.000000


In [36]:
experiment_summary = pd.Series(
    {
        "examples": (
            len(experiment_df)
        ),
        "pairs": (
            experiment_df[
                "pair_id"
            ].nunique()
        ),
        "dialogues": (
            experiment_df[
                "dialogue_id"
            ].nunique()
        ),
        "original_examples": (
            experiment_df[
                "variant"
            ].eq("original").sum()
        ),
        "augmented_examples": (
            experiment_df[
                "variant"
            ].eq("augmented").sum()
        ),
    },
    name="count",
)

experiment_summary.to_frame()

,count
examples,2000
pairs,1000
dialogues,1000
original_examples,1000
augmented_examples,1000


In [37]:
selected_pairs_df[
    "num_augmented_facts"
].value_counts().sort_index()

num_augmented_facts
1.0    606
2.0    305
3.0     78
4.0     11
Name: count, dtype: int64

### 2.2 Final experimental sample

The resulting experimental sample is treated as fixed for all subsequent LLM
evaluation.

The sample contains matched original and augmented persona–dialogue examples.
Both members of a pair contain exactly the same dialogue. The original variant
contains the complete original persona, while the augmented variant contains
the corresponding contradiction-oriented persona produced by the rule-based
augmentation pipeline.

The dataset is generated deterministically using a fixed random seed. No
examples are added, removed, or resampled based on the outputs of the LLM
judges.

In [38]:
EXPERIMENT_DATA_PATH = (
    DATA_DIR
    / "llm_judge_experiment_v2.parquet"
)

experiment_df.to_parquet(
    EXPERIMENT_DATA_PATH,
    index=False,
)

print(
    f"Saved {len(experiment_df):,} examples to "
    f"{EXPERIMENT_DATA_PATH}"
)

Saved 2,000 examples to data/llm_judge_experiment_v2.parquet


## 3. LLM judges

The same experimental examples are evaluated independently by several
instruction-tuned large language models.

Each judge receives:

- the complete persona of the speaker labelled `SELF`,
- the corresponding dialogue,
- the same contradiction-detection instructions.

The model must return a binary decision:

- `CONTRADICTION`
- `NO_CONTRADICTION`

The primary analysis compares decisions between judges rather than treating
the automatically generated expected labels as manually verified ground truth.

### 3.1 Evaluation prompt

The prompt defines contradiction as an incompatibility between the persona
statement and information stated or clearly implied by the speaker labelled
`SELF`.

Statements made by `PARTNER` describe the other participant and must therefore
not be interpreted as information about the persona being evaluated.

The response format is deliberately restricted to a single binary label to
minimize differences caused by free-form generation.

In [39]:
MAIN_PROMPT_ID = "full_persona_v1"


def build_judge_prompt(
    persona: str,
    dialogue: str,
) -> str:
    return f"""You are evaluating whether a dialogue is consistent with a persona.

The persona describes the speaker labelled SELF.

A contradiction exists if at least one statement in the persona cannot be true at the same time as something stated or clearly implied by SELF in the dialogue.

A persona statement does not need to be mentioned in the dialogue. Missing evidence for a persona statement is not a contradiction.

Statements made by PARTNER are not claims made by SELF.

Persona of SELF:
{persona}

Dialogue:
{dialogue}

Does the dialogue contradict any statement in SELF's persona?

Answer with exactly one label:
CONTRADICTION
NO_CONTRADICTION"""

In [44]:
example = experiment_df[
    experiment_df[
        "variant"
    ].eq("original")
].iloc[0]

print(
    build_judge_prompt(
        persona=example[
            "persona_text"
        ],
        dialogue=example[
            "dialogue_text"
        ],
    )
)

You are evaluating whether a dialogue is consistent with a persona.

The persona describes the speaker labelled SELF.

A contradiction exists if at least one statement in the persona cannot be true at the same time as something stated or clearly implied by SELF in the dialogue.

A persona statement does not need to be mentioned in the dialogue. Missing evidence for a persona statement is not a contradiction.

Statements made by PARTNER are not claims made by SELF.

Persona of SELF:
- i need a wheel chair to get around.
- i am a female.
- i have 2 kids.
- i hate tomatoes.

Dialogue:
PARTNER: hello how is your weekend going ?
SELF: good so far . . how about urs ?
PARTNER: its going well . spent the day outside . do you have any pets ?
SELF: no . . i have kids though
PARTNER: how many kids ? i have several different kinds of pets , but no kids .
SELF: 2 kids . . . boy 4 years . . girl 2 years
PARTNER: i prefer spending time with animals lol . your kids are young !
SELF: too young . . they

In [43]:
example = experiment_df[
    experiment_df[
        "variant"
    ].eq("augmented")
].iloc[0]

print(
    build_judge_prompt(
        persona=example[
            "persona_text"
        ],
        dialogue=example[
            "dialogue_text"
        ],
    )
)

You are evaluating whether a dialogue is consistent with a persona.

The persona describes the speaker labelled SELF.

A contradiction exists if at least one statement in the persona cannot be true at the same time as something stated or clearly implied by SELF in the dialogue.

A persona statement does not need to be mentioned in the dialogue. Missing evidence for a persona statement is not a contradiction.

Statements made by PARTNER are not claims made by SELF.

Persona of SELF:
- i do not like fantasizing.
- i like getting packages in the mail.
- i wish magic was real.
- i do not have a daughter.
- i hate christmas.

Dialogue:
PARTNER: how are you doing ? tell me about yourself
SELF: i am doing well , how about you ? i enjoy holidays like christmas
PARTNER: i love christmas too . i like to hike . i am very outdoorsy .
SELF: nice , where do you hike ? i am really daydreamy .
PARTNER: i hike any mountain . but its been hard to hide recently . i am always on my phone .
SELF: why are y

In [51]:
PILOT_PAIRS = 50

pilot_pair_ids = (
    experiment_df[
        ["pair_id"]
    ]
    .drop_duplicates()
    .sample(
        n=PILOT_PAIRS,
        random_state=RANDOM_SEED,
    )
    ["pair_id"]
)

pilot_df = (
    experiment_df[
        experiment_df[
            "pair_id"
        ].isin(pilot_pair_ids)
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(pilot_df) == 2 * PILOT_PAIRS

print(
    f"Pilot: {PILOT_PAIRS} pairs / "
    f"{len(pilot_df)} examples"
)

Pilot: 50 pairs / 100 examples


In [50]:
pd.crosstab(
    selected_pairs_df["num_augmented_facts"],
    selected_pairs_df["num_grounded_augmented_facts"],
)

num_grounded_augmented_facts,1.0,2.0,3.0,4.0
num_augmented_facts,,,,
1.0,606,0,0,0
2.0,2,303,0,0
3.0,0,1,77,0
4.0,0,0,0,11


In [52]:
VALID_LABELS = {
    "CONTRADICTION": 1,
    "NO_CONTRADICTION": 0,
}


def parse_judge_output(
    output: str,
) -> tuple[str | None, int | None]:
    text = output.strip().upper()

    if text == "CONTRADICTION":
        return "CONTRADICTION", 1

    if text == "NO_CONTRADICTION":
        return "NO_CONTRADICTION", 0

    return None, None